In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from src.data.processed_data import get_validated_data
from src.utils.config import FEATURE_ENGINEERED_DATA_PATH

In [3]:
# Load data
data = get_validated_data()
data.set_index("appid", inplace=True)

with open("../outputs/params/eda_outputs.json", "r", encoding="utf-8") as f:
    results = json.load(f)
print(results)

{'popular_genres': ['Indie', 'Casual', 'Adventure', 'Action', 'Simulation', 'Strategy', 'RPG', 'Free To Play', 'Early Access', 'Sports', 'Racing'], 'popular_categories': ['Single-player', 'Family Sharing', 'Steam Achievements', 'Steam Cloud', 'Full controller support', 'Multi-player', 'Partial Controller Support', 'PvP', 'Co-op']}


In [ ]:
# Columns to drop
'''redundant_correlation_features = ["PvP", "Co-op"]
no_modelable_columns = [
    "name",
    "release_year",
    "release_date",
    "genres",
    "categories",
    "developer",
    "publisher",
]'''

In [5]:
# Generate new columns based on existing ones (one hot encoding)
one_hot_genres = data["genres"].str.get_dummies(";").astype("uint8")
one_hot_categories = data["categories"].str.get_dummies(";").astype("uint8")

In [ ]:
# Filter low representation features
one_hot_genres = one_hot_genres.filter(items=results["popular_genres"])
one_hot_categories = one_hot_categories.filter(items=results["popular_categories"])

# Filter redundant correlation features
one_hot_categories = one_hot_categories.drop(columns=results["redundant_correlation_features"])

In [ ]:
# Drop unnecessary columns for the model
data = data.drop(columns=results["no_modelable_columns"])

In [8]:
# Transform columns selected in log1p and sigmoid
data_scaled = data.copy()
data_scaled[["price", "recommendations"]] = np.log1p(data[["price", "recommendations"]])
data_scaled[["price", "recommendations"]] = StandardScaler().fit_transform(
    data_scaled[["price", "recommendations"]]
).astype("float32")
data_scaled.describe()

,price,recommendations
count,6.399500e+04,6.399500e+04
mean,1.108733e-08,-8.583739e-09
std,1.000008e+00,1.000008e+00
min,-1.560754e+00,-3.660991e-01
25%,-8.822340e-01,-3.660991e-01
50%,2.043256e-01,-3.660991e-01
75%,8.027400e-01,-3.660991e-01
max,5.883887e+00,6.100831e+00


In [9]:
# Concat one hot encoded features with transformed selected features
feature_engineered_data = pd.concat(
    [data_scaled, one_hot_genres, one_hot_categories], axis=1
)
feature_engineered_data

,price,recommendations,Indie,Casual,Adventure,Action,Simulation,Strategy,RPG,Free To Play,Early Access,Sports,Racing,Single-player,Family Sharing,Steam Achievements,Steam Cloud,Full controller support,Multi-player,Partial Controller Support
appid,,,,,,,,,,,,,,,,,,,,
3057270,0.024222,-0.366099,1,0,1,1,0,1,1,0,0,0,0,1,1,0,0,0,0,0
3822840,0.604674,-0.366099,1,1,0,0,1,1,0,0,0,0,0,1,1,0,0,0,0,0
3216640,1.040725,-0.366099,1,0,1,0,0,1,0,0,0,0,0,1,1,1,1,1,0,0
2403620,1.651440,-0.366099,1,0,1,1,0,0,0,0,0,0,0,1,1,1,1,1,0,0
1538040,0.024222,-0.366099,1,1,1,1,0,0,1,0,1,0,0,1,1,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3097010,1.172478,-0.366099,1,1,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
3304930,0.204326,-0.366099,1,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,0,0
1461580,1.172478,-0.366099,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0


In [10]:
# save the final dataset to a CSV file
feature_engineered_data.to_parquet(f"{FEATURE_ENGINEERED_DATA_PATH}", index=True)